In [4]:
# ============================================
# TRAIN MODELS DIRECTLY ON REAL DATA (80/20)
# ============================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import recall_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
import joblib

# -----------------------------
# Load real cleaned dataset
# -----------------------------
df = pd.read_csv(r"C:\Users\shiro\OneDrive\Desktop\Python files\Maternal_health_risk_Project\VIz Project\data\Maternal Health Risk Data Set.csv")

X = df.drop("RiskLevel", axis=1)
y = df["RiskLevel"]

# -----------------------------
# Encode labels
# -----------------------------
le = LabelEncoder()
y_enc = le.fit_transform(y)

# -----------------------------
# 80/20 Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

# -----------------------------
# Define models
# -----------------------------
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(max_iter=5000))
    ]),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(
        objective="multi:softprob",
        num_class=len(np.unique(y_enc)),
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42
    )
}

# Voting & Stacking Ensembles
estimators = [
    ("lr", LogisticRegression(max_iter=5000)),
    ("dt", DecisionTreeClassifier()),
    ("rf", RandomForestClassifier()),
    ("gb", GradientBoostingClassifier())
]

models["Voting Ensemble"] = VotingClassifier(
    estimators=estimators,
    voting="soft"
)

models["Stacking Ensemble"] = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=5000)
)

# -----------------------------
# Train + Evaluate + Save Models
# -----------------------------
results = {}

import os
os.makedirs("../models", exist_ok=True)


for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    macro_recall = recall_score(y_test, preds, average="macro")
    results[name] = macro_recall
    
    print(f"Macro Recall: {macro_recall:.4f}")
    print(classification_report(y_test, preds))
    
    # Save model
    joblib.dump(model, f"../models/{name.replace(' ', '_')}.pkl")

# -----------------------------
# Comparison Table
# -----------------------------
comparison_df = pd.DataFrame.from_dict(results, orient="index", columns=["Macro Recall"])
comparison_df = comparison_df.sort_values(by="Macro Recall", ascending=False)

comparison_df



Training Logistic Regression...
Macro Recall: 0.6202
              precision    recall  f1-score   support

           0       0.78      0.78      0.78        55
           1       0.59      0.77      0.67        81
           2       0.49      0.31      0.38        67

    accuracy                           0.62       203
   macro avg       0.62      0.62      0.61       203
weighted avg       0.61      0.62      0.60       203


Training Decision Tree...
Macro Recall: 0.8495
              precision    recall  f1-score   support

           0       0.96      0.93      0.94        55
           1       0.91      0.74      0.82        81
           2       0.70      0.88      0.78        67

    accuracy                           0.84       203
   macro avg       0.86      0.85      0.85       203
weighted avg       0.86      0.84      0.84       203


Training Random Forest...
Macro Recall: 0.8811
              precision    recall  f1-score   support

           0       0.96      0.95

,Macro Recall
Random Forest,0.881149
Voting Ensemble,0.876174
Stacking Ensemble,0.864278
XGBoost,0.864052
Decision Tree,0.849537
Gradient Boosting,0.771878
Logistic Regression,0.620228
